In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system
        
    # add tool_choice forcing:
    params["tool_choice"] = {"type": "tool", "name": "web_search"}

    # Otherwise, the tool_choice will default to {"type": "auto"} 
    # — Claude decides for itself whether a query needs a tool call. 
    # For "What's the best exercise for gaining leg muscle?", 
    # Claude judged this as general knowledge it already has 
    # (not time-sensitive, not something that changed recently), 
    # so it answered directly instead of searching


    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
from IPython.display import display, Markdown


def show_response(message):
    for block in message.content:
        if block.type == "text":
            display(Markdown(block.text))
        elif block.type == "server_tool_use":
            print(f"🔍 Searching: {block.input.get('query')}")
        elif block.type == "web_search_tool_result":
            content = block.content
            if isinstance(content, list):
                print(f"   Found {len(content)} result(s):")
                for result in content:
                    print(f"   - {result.title} ({result.url})")
            else:
                print(f"   Search error: {content}")


In [4]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [5]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
response = chat(messages, tools=[web_search_schema])
show_response(response)

🔍 Searching: best exercises leg muscle growth
   Found 6 result(s):
   - Multi-purpose exerciser (https://image-ppubs.uspto.gov/dirsearch-public/print/downloadPdf/4685671)
   - Weighted-squat exercise machine and belt (https://image-ppubs.uspto.gov/dirsearch-public/print/downloadPdf/11260267)
   - Leg exercise apparatus and method (https://image-ppubs.uspto.gov/dirsearch-public/print/downloadPdf/5181895)
   - Lower body muscle exercise device (https://image-ppubs.uspto.gov/dirsearch-public/print/downloadPdf/9017231)
   - Muscle Strengthening Exercises for the Foot and Ankle: A Scoping Review Exploring Adherence to Best Practice for Optimizing Musculoskeletal Health (https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11967365/)
   - Modular exercise system (https://image-ppubs.uspto.gov/dirsearch-public/print/downloadPdf/11458354)


Based on the search results and fitness science, 

squats are considered by bodybuilders the best leg training exercise

. Here's why squats are so effective:

**Why Squats Are Superior:**
- 

Squats train primarily the muscles of the thighs, hips and buttocks, quadriceps, hamstrings, as well as strengthening the bones, ligaments and insertion of tendons throughout the lower body


- 

Squatting can trigger the release of testosterone and human growth hormone that are vital for muscle growth and to improve muscle mass


- 

Squats are one of the single best exercises for stimulating every single muscle fiber in the lower body while working the core muscles



**Other Highly Effective Leg Exercises:**
While squats are considered the best, other compound movements are also excellent for leg muscle growth:
- **Deadlifts** - Another powerlifting movement that works the entire posterior chain
- **Leg press** - Targets similar muscles to squats with less technical demand
- **Lunges** - Unilateral movement that builds muscle and balance

**Key Training Principles:**
For optimal muscle growth, focus on progressive overload (gradually increasing weight), adequate protein intake, and proper recovery. Using a variety of exercises can help target all leg muscles comprehensively, but if you had to choose one exercise, squats would be the top choice for overall leg muscle development.